[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataphilth/dataphil-lab/blob/main/projects/mlb-shadow-test/notebook.ipynb)

# The MLB shadow test — run it yourself

10,928 predictions from a Monte Carlo model that prices MLB hitter props, each logged
**before its game started** alongside the sportsbook's price, then settled against what
actually happened. June 1 – August 26, 2026.

The model was never bet. It failed a pre-registered gate and this notebook is why.

Run every cell (`Runtime → Run all`, or Ctrl+F9). Nothing to install, nothing to download.

**Full writeup:** https://lab.dataphil.com/mlb-shadow-test/  
**Interactive version:** https://lab.dataphil.com/mlb-shadow-test/explorer.html


## 1. Load the log


In [ ]:
import numpy as np, pandas as pd

URL = "https://raw.githubusercontent.com/dataphilth/dataphil-lab/main/projects/mlb-shadow-test/data/shadow_log.csv"

d = pd.read_csv(URL, parse_dates=['date_placed'])

# edge_pct is stored as a percent in some rows and a fraction in others across the
# log's history; normalise once.
if d.edge_pct.abs().max() > 1.5:
    d['edge_pct'] = d.edge_pct / 100

# win/loss only. 'void' = player scratched or game postponed — excluded, never
# silently counted as a loss.
s = d[d.result.isin(['win','loss'])].copy()
s['won'] = (s.result == 'win').astype(int)

print(f'{len(d):,} logged   {len(s):,} settled   {(d.result=="void").sum():,} void')
print(f'logged through {d.date_placed.max().date()}, settled through {s.date_placed.max().date()}')
s.head(3)[['date_placed','prop_subject','line','american_odds_at_bet','p_model','p_market_devig','result']]


## 2. Is it a good forecaster?

Bin the predictions and compare what the model said to what happened.


In [ ]:
bins = np.arange(0.42, 0.81, 0.06)
cal = (s.assign(b=pd.cut(s.p_model, bins))
         .groupby('b', observed=True)
         .agg(n=('won','size'), predicted=('p_model','mean'), observed=('won','mean'))
         .query('n >= 30').round(4))

print(f"overall: predicted {s.p_model.mean():.4f}   observed {s.won.mean():.4f}   "
      f"Brier {((s.p_model-s.won)**2).mean():.4f}")
cal


Close to the diagonal, mildly overconfident. As a forecaster it works.

## 3. Is it a good *bettor*?

Different question. Betting rewards being right **where the market is wrong** — so sort by
how far the model departed from the de-vigged market price and cut into deciles.

If the model has an edge, the win rate rises left to right.


In [ ]:
def flat_pay(g):
    """Flat-stake payout per leg: +profit if it won, -1 if it lost."""
    o = g.american_odds_at_bet.to_numpy(float)
    profit = np.where(o > 0, o/100, 100/-o)
    return np.where(g.won.to_numpy() == 1, profit, -1.0)

t = s.sort_values(['edge_pct','date_placed']).reset_index(drop=True)
t['decile'] = np.arange(len(t)) * 10 // len(t) + 1

dec = t.groupby('decile').apply(lambda g: pd.Series({
    'n': len(g), 'edge': g.edge_pct.mean(), 'predicted': g.p_model.mean(),
    'observed': g.won.mean(), 'roi_%': flat_pay(g).mean()*100,
}), include_groups=False).round(4)
dec


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,4.5), dpi=110)
ax.plot(dec.index, dec.predicted, 'o--', color='#7d8891', label='model predicted')
ax.plot(dec.index, dec.observed, 'o-', color='#a3332a', lw=2.5, label='actually happened')
ax.set_xticks(dec.index)
ax.set_xlabel('decile of model-vs-market disagreement   (10 = model most bullish)')
ax.set_ylabel('win rate')
ax.set_title('Conviction runs backwards', loc='left')
ax.legend(frameon=False); ax.grid(axis='y', alpha=.3)
for sp in ('top','right'): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.show()


The predicted line climbs. The realised line falls. Where the model was *most* sure the book
was wrong, it went 54% against a 65% forecast.

## 4. Why? A horse race

A model can lose to the market two ways: it knows nothing, or it knows something and is being
used wrong. Opposite remedies, so distinguish them properly.

Regress the outcome on **both** forecasts at once, in log-odds space:

$$\\mathrm{logit}\\,P(\\text{win}) = a + b\\cdot\\mathrm{logit}(p_{\\text{model}}) + c\\cdot\\mathrm{logit}(p_{\\text{market}})$$

If the model is noise, `b` collapses to zero once the market price is in the equation.


In [ ]:
def logit(p): return np.log(p/(1-p))

m = s[s.p_market_devig.between(.01,.99) & s.p_model.between(.01,.99)]
X = np.column_stack([np.ones(len(m)), logit(m.p_model.to_numpy()), logit(m.p_market_devig.to_numpy())])
y = m.won.to_numpy(float)

b = np.zeros(3)                              # Newton-Raphson, so the arithmetic is visible
for _ in range(60):
    p = 1/(1+np.exp(-X@b)); W = np.clip(p*(1-p), 1e-10, None)
    H = X.T @ (X*W[:,None])
    step = np.linalg.solve(H, X.T @ (y-p)); b += step
    if np.abs(step).max() < 1e-11: break
se = np.sqrt(np.diag(np.linalg.inv(H)))

print(pd.DataFrame({'beta': b, 'se': se, 'z': b/se},
                   index=['intercept','logit(p_model)','logit(p_market)']).round(4))
print()
print(f'Brier  model {((m.p_model-y)**2).mean():.4f}   market {((m.p_market_devig-y)**2).mean():.4f}')
print(f'implied weight on the model: {b[1]/(b[1]+b[2]):.1%}')


**The model is not worthless.** It contributes real information at ~2.2 standard errors,
*conditional on already knowing the closing price*. But it earns about **22%** of the combined
weight — while `edge = p_model − p_market` implicitly gives it **50%**.

It was levered ~2.3× past its true strength, and the error scales with the size of the
disagreement. Which is exactly the pattern in section 3.

## 5. Your turn

Change `LAMBDA` and re-run. `1.00` is the original (broken) edge rule; `0.22` is what the
regression measured. Watch how many bets survive.


In [ ]:
LAMBDA = 1.00     # <-- change me: 1.00, 0.75, 0.50, 0.22, 0.00
THRESHOLD = 0.05  # <-- and me: minimum edge required to place the bet

lm, lk = logit(m.p_model.to_numpy()), logit(m.p_market_devig.to_numpy())
p_blend = 1/(1+np.exp(-(LAMBDA*lm + (1-LAMBDA)*lk)))
edge = p_blend - m.p_market_devig.to_numpy()

q = edge >= THRESHOLD
print(f'lambda={LAMBDA:.2f}  threshold={THRESHOLD:.0%}')
print(f'largest edge found anywhere: {edge.max():.1%}')
print(f'{q.sum():,} of {len(m):,} plays clear the bar ({q.sum()/len(m):.1%})')
if q.sum() > 1:
    pay = flat_pay(m[q])
    print(f'  predicted {p_blend[q].mean():.1%}   observed {m.won[q].mean():.1%}')
    print(f'  ROI {pay.mean()*100:+.2f}% +/- {pay.std(ddof=1)/np.sqrt(q.sum())*100:.2f} SE')
else:
    print('  nothing to bet.')


## What this is evidence of

Not a profitable model — the opposite. It's a forward test with the bar set in advance, an
audit trail you just recomputed yourself, and a decision not to ship.

Things worth poking at that aren't done here:

- 10,821 of 10,928 legs are one market (hitter over 0.5 hits, the most efficiently priced
  player prop on the board). Does the pattern hold anywhere else?
- Closing-line value is captured for 964 legs — does it separate anything?
- `shadow_log_totals.csv` in the same folder is a different sub-model that runs the
  *opposite* direction on a much smaller sample. Real, or noise?

— Phil Alexander · [lab.dataphil.com](https://lab.dataphil.com) · [dataphil.com](https://dataphil.com)
